# ENTSO-E clean fetch (DE/LU)
Load forecast, thermal generation/capacity/utilization, net import, activated aFRR/mFRR-up.


In [47]:
# Undo Timedelta patch to allow default behavior (warning may appear)
import pandas as pd
if hasattr(pd, '_orig_Timedelta'):
    pd.Timedelta = pd._orig_Timedelta
import warnings


In [48]:
from entsoe import EntsoePandasClient
import pandas as pd
import requests
import xml.etree.ElementTree as ET
from datetime import datetime
import os
from pathlib import Path

BIDDING_ZONE = '10Y1001A1001A82H'
API_TOKEN = os.environ.get('ENTSOE_API_KEY')
client = EntsoePandasClient(api_key=API_TOKEN)

def fetch_xml_series(params: dict, tz='UTC') -> pd.Series:
    url = 'https://web-api.tp.entsoe.eu/api'
    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    root = ET.fromstring(resp.content)
    rows = []
    for ts in root.findall('.//{*}TimeSeries'):
        for period in ts.findall('.//{*}Period'):
            start = pd.to_datetime(period.findtext('.//{*}timeInterval/{*}start'))
            resolution = period.findtext('.//{*}resolution')
            res = pd.to_timedelta(resolution.replace('PT','').lower())
            for point in period.findall('.//{*}Point'):
                pos = int(point.findtext('.//{*}position')) - 1
                t = start + pos * res
                val = float(point.findtext('.//{*}quantity'))
                rows.append((t, val))
    s = pd.Series(dict(rows)).sort_index()
    return s.tz_localize('UTC').tz_convert(tz)


In [ ]:
# Load forecast via raw A65 (chunked); prefer A01 day-ahead, fallback A31 week-ahead
try:
    load_forecast = query_load_forecast_series(BIDDING_ZONE, start, end, process_type='A01')
    if load_forecast.empty:
        raise ValueError('A01 returned empty')
except Exception as e:
    print('A01 failed, using A31:', e)
    load_forecast = query_load_forecast_series(BIDDING_ZONE, start, end, process_type='A31')
load_forecast.name = 'Ltfc'

# Actual thermal generation (per type, sum thermal PSR)
gen_per_type = client.query_generation(BIDDING_ZONE, start=start, end=end, psr_type=None)
thermal_cols = [t for t in thermal_types if t in gen_per_type.columns]
gen_thermal = gen_per_type[thermal_cols].sum(axis=1).rename('GEN_thermal')

# Installed thermal capacity
installed_per_type = client.query_installed_generation_capacity(
    BIDDING_ZONE, start=start, end=end, psr_type=None
)
cap_cols = [t for t in thermal_types if t in installed_per_type.columns]
cap_thermal = installed_per_type[cap_cols].sum(axis=1).rename('CAP_thermal')

util_thermal = (gen_thermal / cap_thermal).rename('UT_thermal')


In [49]:
# Raw load forecast fetch (A65, day-ahead A01)
def query_load_forecast_raw(zone, start_ts, end_ts):
    params = {
        'securityToken': API_TOKEN,
        'documentType': 'A65',
        'processType': 'A01',
        'outBiddingZone_Domain': zone,
        'periodStart': start_ts.strftime('%Y%m%d%H%M'),
        'periodEnd': end_ts.strftime('%Y%m%d%H%M'),
    }
    return fetch_xml_series(params).rename('Ltfc')


In [50]:
start = pd.Timestamp('2022-01-01', tz='UTC')
end   = pd.Timestamp('2025-12-31', tz='UTC')
thermal_types = [
    'Fossil Brown coal/Lignite','Fossil Coal-derived gas','Fossil Gas',
    'Fossil Hard coal','Fossil Oil','Fossil Oil shale','Fossil Peat',
    'Nuclear','Waste','Other'
]
neighbors = ['10YAT-APG------L','10YBE----------2','10YCH-SWISSGRIDZ','10YCZ-CEPS-----N',
             '10YDK-1--------W','10YDK-2--------M','10YFR-RTE------C','10YNL----------L',
             '10YNO-2--------T','10YPL-AREA-----S','10Y1001A1001A47J']


In [ ]:
# Load forecast (use entsoe-py, fallback A01->A31)
try:
    load_forecast = client.query_load_forecast(BIDDING_ZONE, start=start, end=end, process_type='A01')
except Exception as e:
    print('A01 failed, trying A31:', e)
    load_forecast = client.query_load_forecast(BIDDING_ZONE, start=start, end=end, process_type='A31')
load_forecast.name = 'Ltfc'

# Actual thermal generation (per type, sum thermal PSR)
gen_per_type = client.query_generation(BIDDING_ZONE, start=start, end=end, psr_type=None)
thermal_cols = [t for t in thermal_types if t in gen_per_type.columns]
gen_thermal = gen_per_type[thermal_cols].sum(axis=1).rename('GEN_thermal')

# Installed thermal capacity
installed_per_type = client.query_installed_generation_capacity(
    BIDDING_ZONE, start=start, end=end, psr_type=None
)
cap_cols = [t for t in thermal_types if t in installed_per_type.columns]
cap_thermal = installed_per_type[cap_cols].sum(axis=1).rename('CAP_thermal')

util_thermal = (gen_thermal / cap_thermal).rename('UT_thermal')


In [ ]:
# Net import (sum of neighbor imports minus exports)
def flow_pair(zone_in, zone_out):
    return client.query_crossborder_flows(zone_in, zone_out, start=start, end=end)
flows = []
for n in neighbors:
    f_imp = flow_pair(n, BIDDING_ZONE)
    f_exp = flow_pair(BIDDING_ZONE, n)
    flows.append(f_imp - f_exp)
Xtnet = pd.concat(flows, axis=1).sum(axis=1).rename('Xtnet')


In [ ]:
# Activated aFRR-up and mFRR-up energy (requires periodStart/End)
def activated_reserve(process_type: str, name: str):
    params = {
        'securityToken': API_TOKEN,
        'documentType': 'A83',
        'processType': process_type,
        'businessType': 'A75',
        'controlArea_Domain': BIDDING_ZONE,
        'periodStart': start.strftime('%Y%m%d%H%M'),
        'periodEnd': end.strftime('%Y%m%d%H%M'),
    }
    return fetch_xml_series(params).rename(name)

EtaFRR_up = activated_reserve('A51', 'EtaFRR_up')
EtmFRR_up = activated_reserve('A52', 'EtmFRR_up')


In [ ]:
data = pd.concat([
    load_forecast, gen_thermal, cap_thermal, util_thermal, Xtnet, EtaFRR_up, EtmFRR_up
], axis=1)
data = data.sort_index().resample('1H').ffill()
data.head()


In [ ]:
data_path = Path('data/entsoe_clean.parquet')
data_path.parent.mkdir(parents=True, exist_ok=True)
data.to_parquet(data_path, compression='zstd')
data.describe()


<hr>

In [ ]:
import requests
import os

ENTSOE_API_KEY = os.getenv('ENTSOE_API_KEY') 
base_url = f"https://web-api.tp.entsoe.eu/api?securityToken={ENTSOE_API_KEY}"

start = "202201010000"
end = "202301010000"
BiddingZoneDELU = "10Y1001A1001A82H"
processType_aFRR = "A51" 

headers = {}
payload = {}

# ==============================================================================
# 1. CAPACITY PRICE (Hier MUSS es "Area_Domain" heißen)
# ==============================================================================
print("Lade aFRR Leistungspreise (Capacity)...")
url_capacity = (f"{base_url}&periodStart={start}&periodEnd={end}"
                f"&documentType=A15&processType={processType_aFRR}"
                f"&type_MarketAgreement.Type=A01"
                f"&Area_Domain={BiddingZoneDELU}") # <--- Area_Domain

response_capacity = requests.request("GET", url_capacity, headers=headers, data=payload)

if response_capacity.status_code == 200:
    print("✅ Capacity erfolgreich!")
    with open("afrr_capacity_2022.xml", "w") as f:
        f.write(response_capacity.text)
else:
    print("❌ Fehler Capacity:", response_capacity.text)

# ==============================================================================
# 2. ACTIVATION PRICE (Hier MUSS es "controlArea_Domain" heißen)
# ==============================================================================
print("\nLade aFRR Arbeitspreise (Energy / CBMP)...")
url_energy = (f"{base_url}&periodStart={start}&periodEnd={end}"
              f"&documentType=A84&processType={processType_aFRR}"
              f"&controlArea_Domain={BiddingZoneDELU}") # <--- controlArea_Domain

response_energy = requests.request("GET", url_energy, headers=headers, data=payload)

if response_energy.status_code == 200:
    print("✅ Energy erfolgreich! (Größe:", len(response_energy.text), "Bytes)")
    with open("afrr_energy_cbmp_2022.xml", "w") as f:
        f.write(response_energy.text)
else:
    print("❌ Fehler Energy:", response_energy.text)

Lade aFRR Leistungspreise (Capacity)...
✅ Capacity erfolgreich!

Lade aFRR Arbeitspreise (Energy / CBMP)...
❌ Fehler Energy: <?xml version="1.0" encoding="UTF-8"?>
<Acknowledgement_MarketDocument
	xmlns="urn:iec62325.351:tc57wg16:451-1:acknowledgementdocument:7:0">
	<mRID>bc4bb3f9-647a-4</mRID>
	<createdDateTime>2026-01-23T21:26:40Z</createdDateTime>
	<sender_MarketParticipant.mRID codingScheme="A01">10X1001A1001A450</sender_MarketParticipant.mRID>
	<sender_MarketParticipant.marketRole.type>A32</sender_MarketParticipant.marketRole.type>
	<receiver_MarketParticipant.mRID codingScheme="A01">10X1001A1001A450</receiver_MarketParticipant.mRID>
	<receiver_MarketParticipant.marketRole.type>A39</receiver_MarketParticipant.marketRole.type>
	<received_MarketDocument.createdDateTime>2026-01-23T21:26:40Z</received_MarketDocument.createdDateTime>
	<Reason>
		<code>999</code>
		<text>The provided parameters do not match the dataItem parameters or values.</text>
	</Reason>
</Acknowledgement_MarketDoc